# 第 2 天：本地 Ollama 职位帖分析

## 练习目标（理念）

把职位分析从云端 API（Groq）换成本地 **Ollama** 上的 `deepseek-r1:1.5b`：同一套抓取 + prompt，换推理后端。

- **输入**：职位页 URL
- **后端**：本机 Ollama OpenAI 兼容接口（`/v1`）
- **输出**：结构化职位分析与投递建议

## 和本课 Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 本地模型 | `ollama pull deepseek-r1:1.5b` |
| OpenAI 兼容客户端 | `OpenAI(base_url=..., api_key='ollama')` |
| 同一套 messages | system / user prompt 与 Day 1 同构 |
| 无需云端密钥 | `api_key='ollama'` 仅为占位，真正走本地服务 |

## 怎么跑

1. 先确保本机已安装并启动 Ollama
2. 运行拉取模型单元格（或终端执行同等命令）
3. 从上到下跑完；最后一格已带示例 LinkedIn URL，可改成你自己的职位链接


In [ ]:
# ========== 拉取本地模型：确保 Ollama 已有 deepseek-r1:1.5b ==========
# Jupyter 的 shell magic：在笔记本里执行终端命令（不是 Python 语句）
# 模型名字符串必须与 ollama list 里一致；首次下载可能较久
!ollama pull deepseek-r1:1.5b


In [ ]:
# ========== 导入：抓取工具 + OpenAI 兼容客户端（指向本地 Ollama）==========

# os：环境变量（本格主要用于配合 dotenv）
import os
# requests：下载职位页 HTML
import requests
# BeautifulSoup：HTML → 纯文本
from bs4 import BeautifulSoup
# load_dotenv：加载 .env（本练习即使主要走本地，也保留原加载步骤）
from dotenv import load_dotenv
# OpenAI SDK：这里不用云端，而是把 base_url 指到 Ollama 的 /v1 兼容层
from openai import OpenAI


In [ ]:
# ========== 客户端：指向本机 Ollama 的 OpenAI 兼容端点 ==========

# Ollama 默认在 11434；/v1 提供与 OpenAI Chat Completions 相近的路径
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 加载 .env（override=True 覆盖已有环境变量）；本地推理通常不依赖云密钥
load_dotenv(override=True)
# 创建客户端：base_url 走本地；api_key='ollama' 是占位字符串（Ollama 兼容层常要求非空）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


## 抓取器（Scraper）

与 Day 1 相同的 `fetch_job_text`：把职位 URL 变成去噪纯文本，再交给本地模型分析。


In [ ]:
# ========== 抓取：与云端版同一套路，后端无关 ==========

def fetch_job_text(url):
    # 抓取职位页纯文本
    headers = {"User-Agent": "Mozilla/5.0"}

    # HTTP GET 页面源码
    response = requests.get(url, headers=headers)
    # 解析 HTML DOM
    soup = BeautifulSoup(response.text, "html.parser")

    # 去掉脚本/样式等噪音
    for tag in soup(["script", "style"]):
        tag.decompose()

    # 抽出可见文本
    text = soup.get_text(separator=" ")

    # 折叠空白，便于塞进 prompt
    return " ".join(text.split())


In [ ]:
# ========== system prompt：分析角色与返回格式（英文指令保留，改译会改变行为）==========

system_prompt = """
You are an expert AI Job Analyst.

Your job:
- Analyze job posts clearly and structured
- Extract useful insights
- Be concise and practical

Return format:
1. Job Title
2. Required Skills
3. Experience Level
4. Salary clues (if any)
5. Recommendation (Apply or Skip with reason)

Rules:
- Be direct
- No fluff
- Use bullet points when needed
"""


In [ ]:
# ========== user prompt 前缀：与 job_text 拼接后作为 user message ==========

# 同样含未 .format 的 {job_text} 占位；保持原字符串与拼接方式
user_prompt = """Analyze this job post: {job_text}"""


In [ ]:
# ========== 本地推理：用 ollama 客户端调用 deepseek-r1:1.5b ==========

def analyze_job(job_text):
    # 调用模型分析职位帖（非流式；等整段生成完再返回）
    # 变量名 ollama 是上面创建的 OpenAI 兼容客户端，不是 CLI
    response = ollama.chat.completions.create(
        model="deepseek-r1:1.5b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + (job_text)}
        ]
    )

    # 取出模型回复正文
    return response.choices[0].message.content


In [ ]:
# ========== 主流程：示例 LinkedIn 职位 URL → 抓取 → 本地模型分析 ==========

# URL 字符串保持原样（含查询参数）；可换成你自己的职位链接
url = "https://www.linkedin.com/jobs/view/4399966612/?eBP=CwEAAAGdkeiUgfPQXPJbsAzlxKn8TqSEGdm-PU6_HBfv5TaiVTrX-ZsnOiaFWFOMb4g1oe3Bvxc_Z0Ch2SU9_MwnZh4axmTR0tjnaJ97adnPwEHRSoldY0QiKoUtPu95va22G8HcSYyY-hm9xSNSie1XhilFwJ4qIiloGALkWBCwFG_yt2Tw8z0iuFKZJPeyskI-LyDfZs8m9MucSlWNVZijjc22mSGHfGyWZMDS-k4AZJLLy2WaXTqNGFlfIEkIT3vI1Fg9NJ4rftNLH56UNK2VjUAe0YWxpG-IncGfWMsMI0xQ40gyyc_0csN1B1LxgvPFL7gSxS7Pr9Sqjt2P6zHLxFW7ik5D1CFqKduGk1qzvclKYjSXRkaLKpP2I4F7U4P0Brmm9XN33pZeTdMpPmHDWk9ViDAoFBN59lpg7P158LH1QciWQs4PR-qVPfTsnZCqfgAtkHag9OwuvCtAfBV83lIhQIbx_TXk61qGTWP-LzImZm5eLtA4Tpj4y7l78A&trk=flagship3_search_srp_jobs&refId=XPQru462uw7noXT7ZifDSQ%3D%3D&trackingId=801zqVVlkUBQnJZjDHZtig%3D%3D"
print("\nScraping job post...\n")
# 抓取并清洗页面文本
job_text = fetch_job_text(url)

print("Analyzing with AI...\n")
# 走本地 deepseek-r1:1.5b
result = analyze_job(job_text)

print(result)
